In [1]:
#Import libraries
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score as ss
import pandas as pd 
import numpy as np
import plotly.express as px
from sklearn.cluster import OPTICS
from scipy.spatial import cKDTree
from heapq import heappop, heappush

----
NODES

In [2]:
nodes = pd.read_csv('data/nodes.csv')
nodes

,_id,long,lat
0,366367223,106.629056,10.804243
1,366367233,106.709701,10.771110
2,366367242,106.737189,10.709337
3,366367274,106.760081,10.854489
4,366367285,106.721163,10.804994
...,...,...,...
577962,6202895387,106.647884,10.886330
577963,6202895388,106.649074,10.876678
577964,6203301188,106.700737,10.774919
577965,6203333885,106.699275,10.768892


In [3]:
nodes.drop_duplicates(inplace=True)
nodes.info()
nodes.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577967 entries, 0 to 577966
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   _id     577967 non-null  int64  
 1   long    577967 non-null  float64
 2   lat     577967 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 13.2 MB


_id     0
long    0
lat     0
dtype: int64

----- 
BẢNG SEGMENT


In [4]:
segments = pd.read_csv('data/segments.csv')
segments

,_id,created_at,updated_at,s_node_id,e_node_id,length,street_id,max_velocity,street_level,street_name,street_type
0,0,2020-10-18T13:26:17.365Z,2020-10-18T13:26:17.365Z,373543511,5468660805,114,31096786,80.0,1,Quốc Lộ 1,trunk
1,1,2020-10-18T13:26:17.400Z,2020-10-18T13:26:17.400Z,5468660805,5738158916,9,31096786,80.0,1,Quốc Lộ 1,trunk
2,2,2020-10-18T13:26:17.435Z,2020-10-18T13:26:17.435Z,5738158916,5738158918,23,31096786,80.0,1,Quốc Lộ 1,trunk
3,3,2020-10-18T13:26:17.444Z,2020-10-18T13:26:17.444Z,5738158918,5738158912,66,31096786,80.0,1,Quốc Lộ 1,trunk
4,4,2020-10-18T13:26:17.452Z,2020-10-18T13:26:17.452Z,5738158912,5758104203,127,31096786,80.0,1,Quốc Lộ 1,trunk
...,...,...,...,...,...,...,...,...,...,...,...
84628,84628,2020-10-18T13:30:29.795Z,2020-10-18T13:30:29.795Z,5778600776,411925919,42,658328101,NaN,4,Võ Văn Tần,tertiary
84629,84629,2020-10-18T13:30:29.797Z,2020-10-18T13:30:29.797Z,411925919,3116310151,39,658328101,NaN,4,Võ Văn Tần,tertiary
84630,84630,2020-10-18T13:30:29.799Z,2020-10-18T13:30:29.799Z,3116310151,5778360106,22,658328101,NaN,4,Võ Văn Tần,tertiary
84631,84631,2020-10-18T13:30:29.802Z,2020-10-18T13:30:29.802Z,5778360106,5763168795,37,658328101,NaN,4,Võ Văn Tần,tertiary


In [5]:
segments.drop_duplicates(inplace=True)
segments.info()
segments.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84633 entries, 0 to 84632
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   _id           84633 non-null  int64  
 1   created_at    84633 non-null  object 
 2   updated_at    84633 non-null  object 
 3   s_node_id     84633 non-null  int64  
 4   e_node_id     84633 non-null  int64  
 5   length        84633 non-null  int64  
 6   street_id     84633 non-null  int64  
 7   max_velocity  9871 non-null   float64
 8   street_level  84633 non-null  int64  
 9   street_name   84481 non-null  object 
 10  street_type   84633 non-null  object 
dtypes: float64(1), int64(6), object(4)
memory usage: 7.1+ MB


_id                 0
created_at          0
updated_at          0
s_node_id           0
e_node_id           0
length              0
street_id           0
max_velocity    74762
street_level        0
street_name       152
street_type         0
dtype: int64

-----
BẢNG SEGMENTS_STATUS

In [6]:
segments_status = pd.read_csv('data/segment_status.csv')
segments_status

,_id,updated_at,segment_id,velocity
0,0,2020-07-03T14:55:31.869Z,24845,20
1,1,2020-07-03T15:02:56.048Z,33923,10
2,2,2020-07-04T08:15:52.696Z,33824,5
3,3,2020-07-04T08:15:59.903Z,33824,5
4,4,2020-07-04T08:16:08.201Z,33824,5
...,...,...,...,...
90933,90933,2021-04-22T06:52:39.280Z,52247,1
90934,90934,2021-04-22T06:52:52.501Z,52247,1
90935,90935,2021-04-22T06:53:02.335Z,52247,1
90936,90936,2021-04-22T06:53:14.294Z,52247,1


In [7]:
segments_status.info()
segments_status.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90938 entries, 0 to 90937
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   _id         90938 non-null  int64 
 1   updated_at  90938 non-null  object
 2   segment_id  90938 non-null  int64 
 3   velocity    90938 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 2.8+ MB


_id           0
updated_at    0
segment_id    0
velocity      0
dtype: int64

In [8]:
segments_status['velocity'] = segments_status.groupby('segment_id')['velocity'].transform('mean')
segments_status.drop_duplicates(subset=['segment_id'], inplace=True)
segments_status

,_id,updated_at,segment_id,velocity
0,0,2020-07-03T14:55:31.869Z,24845,16.000000
1,1,2020-07-03T15:02:56.048Z,33923,10.000000
2,2,2020-07-04T08:15:52.696Z,33824,13.000000
29,29,2020-07-04T08:39:14.365Z,56816,25.286232
229,229,2020-07-06T01:09:30.996Z,58241,28.730769
...,...,...,...,...
90429,90429,2021-04-22T05:10:55.269Z,22633,8.000000
90438,90438,2021-04-22T05:12:56.332Z,39486,36.000000
90440,90440,2021-04-22T05:13:18.634Z,52243,9.000000
90442,90442,2021-04-22T05:13:45.168Z,52246,2.000000


----
BẢNG STREETS

In [9]:
streets = pd.read_csv('data/streets.csv')
streets

,_id,level,max_velocity,name,type
0,31096786,1,80.0,Quốc Lộ 1,trunk
1,32575737,4,NaN,NaN,unclassified
2,32575794,4,NaN,Chu Văn An,unclassified
3,32575820,4,NaN,Nguyễn Văn Bá,tertiary
4,32575823,4,NaN,Nguyễn Thị Nhỏ,tertiary
...,...,...,...,...,...
5548,656562464,4,NaN,NaN,unclassified
5549,656564397,4,NaN,NaN,unclassified
5550,656850719,4,NaN,NaN,unclassified
5551,656851094,4,NaN,NaN,unclassified


--------
CREATING BIG TABLE

In [10]:
big_table = segments.merge(nodes, left_on= ["s_node_id"], right_on = ["_id"], how = "left").merge(segments_status, left_on = ["_id_x"], right_on = ["segment_id"], how = "left")
big_table = big_table.merge(nodes, left_on= ["e_node_id"], right_on = ["_id"], how = "left", suffixes=("_xx", "_yy"))

In [11]:
big_table = big_table[["_id_x", "street_id", "street_name", "street_type", "velocity", "max_velocity", "s_node_id", "long_xx", "lat_xx", "e_node_id", "long_yy", "lat_yy"]]
big_table = big_table.rename(columns={"_id_x": "segment_id", 
                          "long_xx": "s_long", 
                          "lat_xx": "s_lat",
                          "long_yy": "e_long",
                          "lat_yy": "e_lat"})
big_table = big_table.dropna(subset=["s_long", "s_lat", "e_long", "e_lat"])
big_table

,segment_id,street_id,street_name,street_type,velocity,max_velocity,s_node_id,s_long,s_lat,e_node_id,e_long,e_lat
0,0,31096786,Quốc Lộ 1,trunk,NaN,80.0,373543511,106.601780,10.727718,5468660805,106.601621,10.726701
1,1,31096786,Quốc Lộ 1,trunk,NaN,80.0,5468660805,106.601621,10.726701,5738158916,106.601607,10.726613
2,2,31096786,Quốc Lộ 1,trunk,NaN,80.0,5738158916,106.601607,10.726613,5738158918,106.601574,10.726401
3,3,31096786,Quốc Lộ 1,trunk,NaN,80.0,5738158918,106.601574,10.726401,5738158912,106.601481,10.725809
4,4,31096786,Quốc Lộ 1,trunk,NaN,80.0,5738158912,106.601481,10.725809,5758104203,106.601277,10.724676
...,...,...,...,...,...,...,...,...,...,...,...,...
84628,84628,658328101,Võ Văn Tần,tertiary,NaN,NaN,5778600776,106.690870,10.777259,411925919,106.690618,10.776970
84629,84629,658328101,Võ Văn Tần,tertiary,NaN,NaN,411925919,106.690618,10.776970,3116310151,106.690381,10.776706
84630,84630,658328101,Võ Văn Tần,tertiary,NaN,NaN,3116310151,106.690381,10.776706,5778360106,106.690243,10.776552
84631,84631,658328101,Võ Văn Tần,tertiary,NaN,NaN,5778360106,106.690243,10.776552,5763168795,106.690018,10.776299


In [12]:
big_table.info()
big_table.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84633 entries, 0 to 84632
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   segment_id    84633 non-null  int64  
 1   street_id     84633 non-null  int64  
 2   street_name   84481 non-null  object 
 3   street_type   84633 non-null  object 
 4   velocity      10027 non-null  float64
 5   max_velocity  9871 non-null   float64
 6   s_node_id     84633 non-null  int64  
 7   s_long        84633 non-null  float64
 8   s_lat         84633 non-null  float64
 9   e_node_id     84633 non-null  int64  
 10  e_long        84633 non-null  float64
 11  e_lat         84633 non-null  float64
dtypes: float64(6), int64(4), object(2)
memory usage: 7.7+ MB


segment_id          0
street_id           0
street_name       152
street_type         0
velocity        74606
max_velocity    74762
s_node_id           0
s_long              0
s_lat               0
e_node_id           0
e_long              0
e_lat               0
dtype: int64

In [13]:
average_max_velocity = big_table[big_table['street_type'] != 'unclassified'].groupby('street_type')['max_velocity'].mean().reset_index().sort_values('max_velocity', ascending=False)
fig = px.bar(average_max_velocity, x='street_type', y='max_velocity', title='Average Max Velocity by Street Type')
fig.show()

In [14]:
velocity_dict = average_max_velocity.set_index('street_type')['max_velocity'].to_dict()
big_table['max_velocity'] = big_table['max_velocity'].fillna(big_table['street_type'].map(velocity_dict))
big_table.dropna(subset=['max_velocity'], inplace=True)
print("Null max_velocity",big_table['max_velocity'].isna().sum())

Null max_velocity 0


In [15]:
known_streets = big_table.dropna(subset=['street_name'])
tree = cKDTree(known_streets[['s_lat', 's_long']])
missing_streets = big_table[big_table['street_name'].isna()]
distances, indices = tree.query(missing_streets[['s_lat', 's_long']])
big_table.loc[big_table['street_name'].isna(), 'street_name'] = known_streets.iloc[indices]['street_name'].values
print("Null street_name",big_table['street_name'].isna().sum())

Null street_name 0


In [16]:
average_velocity_street_name = big_table.groupby('street_name')['velocity'].mean().reset_index()
average_velocity_street_name = average_velocity_street_name.dropna(subset=['velocity']).sort_values(by='velocity', ascending=False)
fig = px.bar(average_velocity_street_name, x='street_name', y='velocity', title='Average Velocity by Street Name')
fig.show()

In [17]:
big_table['velocity'] = big_table.groupby('street_name')['velocity'].transform(pd.Series.fillna, big_table.groupby('street_name')['velocity'].transform('mean'))
print("Null velocity",big_table['velocity'].isna().sum())
big_table.dropna(subset=['velocity'], inplace=True)
big_table.loc[big_table['velocity'] > big_table['max_velocity'], 'velocity'] = big_table['max_velocity']

Null velocity 16119


In [18]:
big_table['slow_traffic'] = np.where(big_table['velocity'] >= 20, 0, 1)
big_table

,segment_id,street_id,street_name,street_type,velocity,max_velocity,s_node_id,s_long,s_lat,e_node_id,e_long,e_lat,slow_traffic
0,0,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,373543511,106.601780,10.727718,5468660805,106.601621,10.726701,0
1,1,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5468660805,106.601621,10.726701,5738158916,106.601607,10.726613,0
2,2,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158916,106.601607,10.726613,5738158918,106.601574,10.726401,0
3,3,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158918,106.601574,10.726401,5738158912,106.601481,10.725809,0
4,4,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158912,106.601481,10.725809,5758104203,106.601277,10.724676,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
84628,84628,658328101,Võ Văn Tần,tertiary,28.783333,47.917182,5778600776,106.690870,10.777259,411925919,106.690618,10.776970,0
84629,84629,658328101,Võ Văn Tần,tertiary,28.783333,47.917182,411925919,106.690618,10.776970,3116310151,106.690381,10.776706,0
84630,84630,658328101,Võ Văn Tần,tertiary,28.783333,47.917182,3116310151,106.690381,10.776706,5778360106,106.690243,10.776552,0
84631,84631,658328101,Võ Văn Tần,tertiary,28.783333,47.917182,5778360106,106.690243,10.776552,5763168795,106.690018,10.776299,0


In [19]:
traffic_coords = pd.concat([big_table[['s_lat', 's_long']].rename(columns={'s_lat': 'lat', 's_long': 'long'}), 
                            big_table[['e_lat', 'e_long']].rename(columns={'e_lat': 'lat', 'e_long': 'long'})], 
                           axis=0).drop_duplicates().reset_index(drop=True)
traffic_coords

,lat,long
0,10.727718,106.601780
1,10.726701,106.601621
2,10.726613,106.601607
3,10.726401,106.601574
4,10.725809,106.601481
...,...,...
34485,10.728596,106.698581
34486,10.874513,106.809287
34487,10.794158,106.677779
34488,10.794735,106.677816


----------
TRAIN 

In [20]:
def loop_dbscan(min_clusters, max_clusters): 
    best_score = -1
    best_eps = 0.1
    best_min_pts = 3
    traffic_coords_copy = traffic_coords.copy()
    for min_pts in range(4, 10):
        for eps in np.linspace(start=0.001, stop=0.0001, num=10):
            model = DBSCAN(eps=eps, min_samples=min_pts).fit(traffic_coords_copy)
            traffic_coords['dbscan_cluster_labels'] = model.labels_
            num_clusters = len(set(traffic_coords['dbscan_cluster_labels'])) - (1 if -1 in traffic_coords['dbscan_cluster_labels'] else 0)
            if num_clusters >= min_clusters and max_clusters >= num_clusters:
                score = ss(traffic_coords_copy, traffic_coords['dbscan_cluster_labels'])
                noise = list(traffic_coords['dbscan_cluster_labels']).count(-1)
                print("Training with eps =", eps, "and min_samples =", min_pts, "found", num_clusters, "clusters", "with a silhouette score of", score, "and", noise, "noise points")
                if score > best_score:
                    best_score = score
                    best_eps = eps
                    best_min_pts = min_pts
                   
    return best_eps, best_min_pts

best_eps, best_min_samples = loop_dbscan(500,1000)


def loop_optics(min_clusters, max_clusters):
    best_score = -1
    best_min_pts = None

    for min_pts in range(4, 10):
        model = OPTICS(min_samples=min_pts).fit(traffic_coords[['lat', 'long']])
        traffic_coords['optic_cluster_labels'] = model.labels_
        num_clusters = len(set(traffic_coords['optic_cluster_labels'])) - (1 if -1 in traffic_coords['optic_cluster_labels'] else 0)
        if min_clusters <= num_clusters <= max_clusters and num_clusters > 1:
            score = ss(traffic_coords, traffic_coords['optic_cluster_labels'])
            noise = list(traffic_coords['optic_cluster_labels']).count(-1)
            print("Training with min_samples =", min_pts, "found", num_clusters, "clusters", "with a silhouette score of", score, "and", noise)
            if score > best_score:
                best_score = score
                best_min_pts = min_pts
    return best_min_pts, best_score

best_min_pts, best_score = loop_optics(2, 4000)

In [21]:
model_dbscan = DBSCAN(eps= 0.0006, min_samples = 4).fit(traffic_coords[['lat', 'long']])
traffic_coords['dbscan_cluster_labels'] = model_dbscan.labels_
model_optics = OPTICS(min_samples = 4).fit(traffic_coords[['lat', 'long']])
traffic_coords['optic_cluster_labels'] = model_optics.labels_

In [22]:

fig_optics = px.scatter_mapbox(
    traffic_coords,
    lat="lat",
    lon="long",
    color="optic_cluster_labels",
    color_continuous_scale="Viridis",
    title="OPTICS Clustering Visualization",
    zoom=12,
    height=600
)

# Set the map style
fig_optics.update_layout(mapbox_style="carto-positron")

# Show the plot
fig_optics.show()

In [23]:
fig_dbscan = px.scatter_mapbox(
    traffic_coords,
    lat="lat",
    lon="long",
    color="dbscan_cluster_labels",
    color_continuous_scale="Viridis",
    title="DBSCAN Clustering Visualization",
    zoom=12,
    height=600
)

# Set the map style
fig_dbscan.update_layout(mapbox_style="carto-positron")

# Show the plot
fig_dbscan.show()

In [24]:
big_table_2 = big_table.merge(traffic_coords, left_on=['s_lat', 's_long'], right_on=['lat', 'long'], how='left')
big_table_2 = big_table_2.drop(columns=['lat', 'long'])
big_table_2.head(1000)

,segment_id,street_id,street_name,street_type,velocity,max_velocity,s_node_id,s_long,s_lat,e_node_id,e_long,e_lat,slow_traffic,dbscan_cluster_labels,optic_cluster_labels
0,0,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,373543511,106.601780,10.727718,5468660805,106.601621,10.726701,0,0,0
1,1,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5468660805,106.601621,10.726701,5738158916,106.601607,10.726613,0,0,2
2,2,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158916,106.601607,10.726613,5738158918,106.601574,10.726401,0,0,2
3,3,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158918,106.601574,10.726401,5738158912,106.601481,10.725809,0,0,-1
4,4,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158912,106.601481,10.725809,5758104203,106.601277,10.724676,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1615,32577116,Thích Quảng Đức,tertiary,42.130769,47.917182,3654066932,106.683699,10.804223,5763293916,106.683640,10.804272,0,68,1058
996,1616,32577116,Thích Quảng Đức,tertiary,42.130769,47.917182,3654066932,106.683699,10.804223,710640671,106.683770,10.804151,0,68,1058
997,1617,32577116,Thích Quảng Đức,tertiary,47.917182,47.917182,710640671,106.683770,10.804151,3654066932,106.683699,10.804223,0,68,1058
998,1618,32577116,Thích Quảng Đức,tertiary,42.130769,47.917182,710640671,106.683770,10.804151,5763160800,106.683900,10.804041,0,68,1058


In [25]:
big_table_2.to_csv('big_table_2.csv', index=False)

In [26]:
def heuristic_dbscan(node, goal):
    node_coords = big_table_2[big_table_2['dbscan_cluster_labels'] == node][['s_lat', 's_long']].mean()
    goal_coords = big_table_2[big_table_2['dbscan_cluster_labels'] == goal][['s_lat', 's_long']].mean()
    return abs(node_coords['s_lat'] - goal_coords['s_lat']) + abs(node_coords['s_long'] - goal_coords['s_long'])

def get_neighbors_dbscan(node, cluster_weights, distance_threshold=0.02):
    node_coords = big_table_2[big_table_2['dbscan_cluster_labels'] == node][['s_lat', 's_long']].mean()
    neighbors = []
    for cluster in cluster_weights.keys():
        if cluster != node and cluster in big_table_2['dbscan_cluster_labels'].unique():
            cluster_coords = big_table_2[big_table_2['dbscan_cluster_labels'] == cluster][['s_lat', 's_long']].mean()
            distance = abs(node_coords['s_lat'] - cluster_coords['s_lat']) + abs(node_coords['s_long'] - cluster_coords['s_long'])
            if distance <= distance_threshold:
                neighbors.append(cluster)
    #print("Current node:", node)
    #print("Neighbors found:", neighbors)
    return neighbors

def a_star_dbscan(start, goal):
    cluster_weights = big_table_2[big_table_2['slow_traffic'] == 1].groupby('dbscan_cluster_labels').size().to_dict()
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {start: 0}
    f_score = {start: heuristic_dbscan(start, goal)}
    path = []
    while open_set:
        current = heappop(open_set)[1]
        if current == goal:
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]
        for neighbor in get_neighbors_dbscan(current, cluster_weights):
            tentative_g_score = g_score[current] + cluster_weights.get(neighbor, 0)
            if neighbor not in g_score or tentative_g_score < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = tentative_g_score + heuristic_dbscan(neighbor, goal)
                if neighbor not in [i[1] for i in open_set]:
                    heappush(open_set, (f_score[neighbor], neighbor))
    return None 

In [ ]:
def heuristic_optics(node, goal):
    node_coords = big_table_2[big_table_2['optic_cluster_labels'] == node][['s_lat', 's_long']].mean()
    goal_coords = big_table_2[big_table_2['optic_cluster_labels'] == goal][['s_lat', 's_long']].mean()
    return abs(node_coords['s_lat'] - goal_coords['s_lat']) + abs(node_coords['s_long'] - goal_coords['s_long'])

def get_neighbors_optics(node, cluster_weights, distance_threshold=0.02):
    node_coords = big_table_2[big_table_2['optic_cluster_labels'] == node][['s_lat', 's_long']].mean()
    neighbors = []
    for cluster in cluster_weights.keys():
        if cluster != node and cluster in big_table_2['optic_cluster_labels'].unique():
            cluster_coords = big_table_2[big_table_2['optic_cluster_labels'] == cluster][['s_lat', 's_long']].mean()
            distance = abs(node_coords['s_lat'] - cluster_coords['s_lat']) + abs(node_coords['s_long'] - cluster_coords['s_long'])
            if distance <= distance_threshold:
                neighbors.append(cluster)
    #print("Current node:", node)
    #print("Neighbors found:", neighbors)
    return neighbors

def a_star_dbscan(start, goal):
    cluster_weights = big_table_2[big_table_2['slow_traffic'] == 1].groupby('optic_cluster_labels').size().to_dict()
    open_set = []
    heappush(open_set, (0, start))
    came_from = {}
    g_score = {start: 0}
    f_score = {start: heuristic_dbscan(start, goal)}
    path = []
    while open_set:
        current = heappop(open_set)[1]
        if current == goal:
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]
        for neighbor in get_neighbors_dbscan(current, cluster_weights):
            tentative_g_score = g_score[current] + cluster_weights.get(neighbor, 0)
            if neighbor not in g_score or tentative_g_score < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = tentative_g_score + heuristic_dbscan(neighbor, goal)
                if neighbor not in [i[1] for i in open_set]:
                    heappush(open_set, (f_score[neighbor], neighbor))
    return None 

CASE 1

In [28]:
start_cluster = 377
end_cluster = 677
optimized_path_dbscan = a_star_dbscan(start_cluster, end_cluster)
print("Optimized Path (DBSCAN):", optimized_path_dbscan)


Optimized Path (DBSCAN): [377, 763, 30, 1414, 905, 677]


In [29]:
start_cluster = 2932
end_cluster = 376
optimized_path_optics = a_star_optics(start_cluster, end_cluster)
print("Optimized Path (OPTICS):", optimized_path_optics)

KeyboardInterrupt: 

CASE 2

In [30]:
start_cluster = 1234
end_cluster = 0
optimized_path_dbscan = a_star_dbscan(start_cluster, end_cluster)
print("Optimized Path (DBSCAN):", optimized_path_dbscan)

Optimized Path (DBSCAN): [1234, 1499, 385, 44, 1540, 318, 275, 1361, 1162, 237, 236, 158, 0]


In [ ]:
start_cluster = 3420
end_cluster = 31
optimized_path_optics = a_star_optics(start_cluster, end_cluster)
print("Optimized Path (OPTICS):", optimized_path_optics)

CASE 3

In [31]:
start_cluster = 572
end_cluster = 56
optimized_path_dbscan = a_star_dbscan(start_cluster, end_cluster)
print("Optimized Path (DBSCAN):", optimized_path_dbscan)

Optimized Path (DBSCAN): [572, 1182, 943, 1465, 942, 56]


In [ ]:
start_cluster = 1832
end_cluster = 2009
optimized_path_optics = a_star_optics(start_cluster, end_cluster)
print("Optimized Path (OPTICS):", optimized_path_optics)

In [ ]:
import folium

# Define start_coords and end_coords based on optimized_path
start_coords = traffic_coords[traffic_coords['dbscan_cluster_labels'] == optimized_path_dbscan[0]]
end_coords = traffic_coords[traffic_coords['dbscan_cluster_labels'] == optimized_path_dbscan[-1]]

# Create a map centered around the start cluster
map_folium = folium.Map(location=[start_coords['lat'].mean(), start_coords['long'].mean()], zoom_start=14)

# Add markers for the start and end clusters
folium.Marker(
    location=[start_coords['lat'].mean(), start_coords['long'].mean()],
    popup="Start Cluster",
    icon=folium.Icon(color="green")
).add_to(map_folium)

folium.Marker(
    location=[end_coords['lat'].mean(), end_coords['long'].mean()],
    popup="End Cluster",
    icon=folium.Icon(color="red")
).add_to(map_folium)

# Add the path generated by the a_star algorithm with arrows
for i in range(len(optimized_path_dbscan) - 1):
    start = traffic_coords[traffic_coords['dbscan_cluster_labels'] == optimized_path_dbscan[i]][['lat', 'long']].mean()
    end = traffic_coords[traffic_coords['dbscan_cluster_labels'] == optimized_path_dbscan[i + 1]][['lat', 'long']].mean()
    folium.PolyLine(
        [(start['lat'], start['long']), (end['lat'], end['long'])],
        color="red",
        weight=2.5,
        tooltip="Path"
    ).add_to(map_folium)
    # Add arrowheads manually by adding a marker at the midpoint of the line
    midpoint = [(end['lat'] + start['lat']) / 2, (end['long'] + start['long']) / 2]
    folium.RegularPolygonMarker(
        location=midpoint,
        number_of_sides=3,
        radius=6,
        rotation=180,  # Rotate the arrowhead to point in the correct direction
        color="red",
        fill=True,
        fill_color="red"
    ).add_to(map_folium)

# Add data points for the clusters in the path
for cluster in optimized_path_dbscan:
    cluster_points = traffic_coords[traffic_coords['dbscan_cluster_labels'] == cluster]
    for _, row in cluster_points.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['long']],
            radius=3,
            color="blue",
            fill=True,
            fill_color="blue",
            fill_opacity=0.7
        ).add_to(map_folium)

# Display the map
map_folium

In [ ]:
import folium

# Define start_coords and end_coords based on optimized_path
start_coords = traffic_coords[traffic_coords['optic_cluster_labels'] == optimized_path_optics[0]]
end_coords = traffic_coords[traffic_coords['optic_cluster_labels'] == optimized_path_optics[-1]]

# Create a map centered around the start cluster
map_folium = folium.Map(location=[start_coords['lat'].mean(), start_coords['long'].mean()], zoom_start=14)

# Add markers for the start and end clusters
folium.Marker(
    location=[start_coords['lat'].mean(), start_coords['long'].mean()],
    popup="Start Cluster",
    icon=folium.Icon(color="green")
).add_to(map_folium)

folium.Marker(
    location=[end_coords['lat'].mean(), end_coords['long'].mean()],
    popup="End Cluster",
    icon=folium.Icon(color="red")
).add_to(map_folium)

# Add the path generated by the a_star algorithm with arrows
for i in range(len(optimized_path_optics) - 1):
    start = traffic_coords[traffic_coords['optic_cluster_labels'] == optimized_path_optics[i]][['lat', 'long']].mean()
    end = traffic_coords[traffic_coords['optic_cluster_labels'] == optimized_path_optics[i + 1]][['lat', 'long']].mean()
    folium.PolyLine(
        [(start['lat'], start['long']), (end['lat'], end['long'])],
        color="red",
        weight=2.5,
        tooltip="Path"
    ).add_to(map_folium)
    # Add arrowheads manually by adding a marker at the midpoint of the line
    midpoint = [(end['lat'] + start['lat']) / 2, (end['long'] + start['long']) / 2]
    folium.RegularPolygonMarker(
        location=midpoint,
        number_of_sides=3,
        radius=6,
        rotation=180,  # Rotate the arrowhead to point in the correct direction
        color="red",
        fill=True,
        fill_color="red"
    ).add_to(map_folium)

# Add data points for the clusters in the path
for cluster in optimized_path_optics:
    cluster_points = traffic_coords[traffic_coords['optic_cluster_labels'] == cluster]
    for _, row in cluster_points.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['long']],
            radius=3,
            color="blue",
            fill=True,
            fill_color="blue",
            fill_opacity=0.7
        ).add_to(map_folium)

# Display the map
map_folium